In [28]:
from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [29]:
# Membaca berkas CSV lokal menjadi Spark DataFrame
# header=True    -> baris pertama dianggap nama kolom
# inferSchema=True -> Spark otomatis menebak tipe data tiap kolom (angka, teks, dst.)
df = spark.read.csv("transaksi_september_2026.csv", header=True, inferSchema=True)

print("Tipe objek:", type(df))
df.printSchema()

Tipe objek: <class 'pyspark.sql.dataframe.DataFrame'>
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)



In [30]:
# Menampilkan beberapa baris pertama — mirip df.head() di pandas, namun disebut show()
df.show(5)

# Menghitung jumlah baris — mirip len(df) di pandas
print("Jumlah baris:", df.count())

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|
|ORD-3004|2026-09-10 00:00:00|        Rumah Tangga|Yogyakarta|          10|       60000|         E-Wallet|   4.0|
+--------+-------------------+--------------------+----------+------------+------------+

In [31]:
df.show(10, truncate=False)

+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+
|order_id|tanggal            |kategori              |kota      |unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|Rumah Tangga          |Yogyakarta|3           |90000       |COD              |4.0   |
|ORD-3001|2026-09-04 00:00:00|Makanan & Minuman     |Solo      |3           |200000      |E-Wallet         |5.0   |
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecantikan|Semarang  |8           |60000       |E-Wallet         |3.0   |
|ORD-3003|2026-09-09 00:00:00|Makanan & Minuman     |Semarang  |6           |350000      |Transfer Bank    |4.0   |
|ORD-3004|2026-09-10 00:00:00|Rumah Tangga          |Yogyakarta|10          |60000       |E-Wallet         |4.0   |
|ORD-3005|2026-09-09 00:00:00|Fashion               |Purworejo |5       

In [32]:
# b
from pyspark.sql.functions import col

df.filter(col("rating").isNull()).count()

204

In [33]:
df = df.na.fill({"rating": 30})

In [34]:
df.filter(col("rating").isNull()).count()

0

In [35]:
# Menambahkan total pendapatan
df = df.withColumn("total_pendapatan", col("unit_terjual")*col("harga_satuan"))

In [36]:
df.select(
    "order_id",
    "unit_terjual",
    "harga_satuan",
    "total_pendapatan"
).show(10)

+--------+------------+------------+----------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|
+--------+------------+------------+----------------+
|ORD-3000|           3|       90000|          270000|
|ORD-3001|           3|      200000|          600000|
|ORD-3002|           8|       60000|          480000|
|ORD-3003|           6|      350000|         2100000|
|ORD-3004|          10|       60000|          600000|
|ORD-3005|           5|       20000|          100000|
|ORD-3006|           2|       20000|           40000|
|ORD-3007|           8|       90000|          720000|
|ORD-3008|           7|       20000|          140000|
|ORD-3009|          10|       90000|          900000|
+--------+------------+------------+----------------+
only showing top 10 rows



In [37]:
# Menambahkan tier_transaksi
from pyspark.sql.functions import col
from pyspark.sql.functions import when

spark = SparkSession.builder.appName("Transaksi").getOrCreate()

df = df.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000,"Besar")
    .otherwise("Kecil"))
df.show()

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

In [38]:
from pyspark.sql.functions import sum 
hasil_kategori = df.groupBy("kategori").agg(sum("total_pendapatan").alias("total_pendapatan")).orderBy("total_pendapatan", ascending=False).show(1)

+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row



In [39]:
from pyspark.sql.functions import col, desc
hasil_kota = df.filter(col("tier_transaksi") == "Besar").groupBy("kota").count().orderBy(desc("count")).show(1)

+----+-----+
|kota|count|
+----+-----+
|Solo|   92|
+----+-----+
only showing top 1 row



In [40]:
from pyspark.sql.functions import avg
hasil_rating = df.groupBy("metode_pembayaran").agg(avg("rating").alias("rata_rata_rating")).orderBy("metode_pembayaran").show()

+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|9.111553784860558|
|         E-Wallet|            9.412|
|     Kartu Kredit|9.898373983739837|
|    Transfer Bank|9.268774703557312|
+-----------------+-----------------+



In [41]:
#d
output_path = "hasil_transaksi_september_2026_2026"
df.write.mode("overwrite").option("header", True).csv(output_path)

In [42]:
df_hasil = spark.read.option("header", True).option("inferSchema", True).csv(output_path)
print("jumlah baris hasil:", df_hasil.count())

jumlah baris hasil: 1000


In [43]:
df_hasil.show(10, truncate=False)

+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|tanggal            |kategori              |kota      |unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|Rumah Tangga          |Yogyakarta|3           |90000       |COD              |4.0   |270000          |Kecil         |
|ORD-3001|2026-09-04 00:00:00|Makanan & Minuman     |Solo      |3           |200000      |E-Wallet         |5.0   |600000          |Besar         |
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecantikan|Semarang  |8           |60000       |E-Wallet         |3.0   |480000          |Kecil         |
|ORD-3003|2026-09-09 00:00:00|Makanan & Minuman     |Semarang  |6           |350000      |Transfer Bank    |4.0 

In [44]:
!ls -l hasil_transaksi_september_2026_2026

total 96
-rw-r--r-- 1 vey vey 97500 Sep 13 06:25 part-00000-046c3974-0b44-4411-972b-777b012fe760-c000.csv
-rw-r--r-- 1 vey vey     0 Sep 13 06:25 _SUCCESS


In [45]:
!hdfs dfs -put hasil_transaksi_september_2026_2026 /user/mahasiswa/tugas4/

put: `/user/mahasiswa/tugas4/hasil_transaksi_september_2026_2026/_SUCCESS': File exists


In [46]:
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_transaksi_september_2026_2026

Found 4 items
-rw-r--r--   1 vey supergroup          0 2026-09-10 11:55 /user/mahasiswa/tugas4/hasil_transaksi_september_2026_2026/_SUCCESS
-rw-r--r--   1 vey supergroup      97500 2026-09-13 06:25 /user/mahasiswa/tugas4/hasil_transaksi_september_2026_2026/part-00000-046c3974-0b44-4411-972b-777b012fe760-c000.csv
-rw-r--r--   1 vey supergroup      97500 2026-09-13 06:25 /user/mahasiswa/tugas4/hasil_transaksi_september_2026_2026/part-00000-280488be-96d3-4515-906f-ab08cd7436ef-c000.csv
-rw-r--r--   1 vey supergroup     104672 2026-09-10 11:55 /user/mahasiswa/tugas4/hasil_transaksi_september_2026_2026/part-00000-d2d5257c-d79e-48d9-9b1c-44fa6abd5d0e-c000.csv


In [50]:
# eksplorasi
df.filter(col("rating") == 5) \
  .select("order_id", "kategori", "kota", "rating", "total_pendapatan") \
  .show(10, truncate=False)

+--------+----------------------+----------+------+----------------+
|order_id|kategori              |kota      |rating|total_pendapatan|
+--------+----------------------+----------+------+----------------+
|ORD-3001|Makanan & Minuman     |Solo      |5.0   |600000          |
|ORD-3006|Makanan & Minuman     |Yogyakarta|5.0   |40000           |
|ORD-3008|Fashion               |Semarang  |5.0   |140000          |
|ORD-3012|Kesehatan & Kecantikan|Solo      |5.0   |1000000         |
|ORD-3013|Makanan & Minuman     |Yogyakarta|5.0   |1600000         |
|ORD-3015|Elektronik            |Purworejo |5.0   |1600000         |
|ORD-3018|Fashion               |Semarang  |5.0   |990000          |
|ORD-3020|Olahraga              |Magelang  |5.0   |360000          |
|ORD-3021|Olahraga              |Kebumen   |5.0   |625000          |
|ORD-3027|Fashion               |Kebumen   |5.0   |630000          |
+--------+----------------------+----------+------+----------------+
only showing top 10 rows



In [51]:
df.groupBy("metode_pembayaran") \
  .count() \
  .orderBy(col("count").desc()) \
  .show()

+-----------------+-----+
|metode_pembayaran|count|
+-----------------+-----+
|    Transfer Bank|  253|
|              COD|  251|
|         E-Wallet|  250|
|     Kartu Kredit|  246|
+-----------------+-----+



In [52]:
from pyspark.sql.functions import avg, sum, col

df.groupBy("kategori") \
  .agg(
      avg("rating").alias("rata_rata_rating"),
      sum("total_pendapatan").alias("total_pendapatan")
  ) \
  .orderBy(col("rata_rata_rating").desc()) \
  .show()

+--------------------+-----------------+----------------+
|            kategori| rata_rata_rating|total_pendapatan|
+--------------------+-----------------+----------------+
|            Olahraga|10.65680473372781|       126650000|
|   Makanan & Minuman|9.896969696969697|       131890000|
|Kesehatan & Kecan...|9.779661016949152|       128595000|
|        Rumah Tangga|9.544444444444444|       138665000|
|          Elektronik|8.763157894736842|       110295000|
|             Fashion| 7.67515923566879|       124075000|
+--------------------+-----------------+----------------+



In [53]:
from pyspark.sql.functions import count, sum, col

df.groupBy("kota") \
  .agg(
      count("order_id").alias("jumlah_transaksi"),
      sum("total_pendapatan").alias("total_pendapatan")
  ) \
  .orderBy(
      col("jumlah_transaksi").desc(),
      col("total_pendapatan").desc()
  ) \
  .show()

+----------+----------------+----------------+
|      kota|jumlah_transaksi|total_pendapatan|
+----------+----------------+----------------+
|      Solo|             189|       143950000|
|Yogyakarta|             175|       122045000|
|   Kebumen|             164|       122615000|
|  Magelang|             162|       129860000|
|  Semarang|             158|       115455000|
| Purworejo|             152|       126245000|
+----------+----------------+----------------+



In [54]:
from pyspark.sql.functions import avg, sum, count, col

df.groupBy("tier_transaksi") \
  .agg(
      count("order_id").alias("jumlah_transaksi"),
      sum("total_pendapatan").alias("total_pendapatan"),
      avg("total_pendapatan").alias("rata_rata_pendapatan"),
      avg("rating").alias("rata_rata_rating")
  ) \
  .orderBy(col("total_pendapatan").desc()) \
  .show()

+--------------+----------------+----------------+--------------------+------------------+
|tier_transaksi|jumlah_transaksi|total_pendapatan|rata_rata_pendapatan|  rata_rata_rating|
+--------------+----------------+----------------+--------------------+------------------+
|         Besar|             454|       633820000|   1396079.295154185|10.002202643171806|
|         Kecil|             546|       126350000|   231410.2564102564| 8.935897435897436|
+--------------+----------------+----------------+--------------------+------------------+



In [55]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
